# Evaluation of IDF of hourly CPM emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import string
import xarray as xr

from mlde_analysis.idf import calc_spells, select_spells, calc_distn, BINS, NDURS, NBINS, DURATIONS

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## IDF

In [ ]:
%%time

var = "pr"

sub_reg_dist = np.zeros((NDURS,NBINS),dtype=np.long)

da = TARGET_DAS["pr"]

point_ts_da = da.isel(grid_latitude=[31], grid_longitude=[31], ensemble_member=[0])
# for (lat, lon), point_ts_da in da.groupby(["grid_latitude","grid_longitude"]):
for (sesaon, year), sy_point_ts_da in point_ts_da.groupby(["time.season", "time.year"]):
    spells = calc_spells(sy_point_ts_da.squeeze().values)
    # season_timeseries = sy_point_ts_da["time.season"].values
    # spells_select = select_spells(spells,season_timeseries)
    hist  = calc_distn(spells)
    sub_reg_dist[:,:] = sub_reg_dist[:,:] + hist[0]
    
shw = plt.imshow(sub_reg_dist)
shw.axes.set_title("CPM")
bar = plt.colorbar(shw)
plt.show()

In [ ]:
%%time

var = "pr"

sub_reg_dist = np.zeros((NDURS,NBINS),dtype=np.long)

da = PRED_DAS["pr"]
point_ts_da = da.isel(grid_latitude=[31], grid_longitude=[31], ensemble_member=[0])
for model, model_point_ts_da in point_ts_da.groupby("model"):
    # for (lat, lon), point_ts_da in da.groupby(["grid_latitude","grid_longitude"]):
    for (sesaon, year), sy_model_point_ts_da in model_point_ts_da.groupby(["time.season", "time.year"]):
        spells = calc_spells(sy_model_point_ts_da.squeeze().values)
        # season_timeseries = sy_point_ts_da["time.season"].values
        # spells_select = select_spells(spells,season_timeseries)
        hist  = calc_distn(spells)
        sub_reg_dist[:,:] = sub_reg_dist[:,:] + hist[0]

    shw = plt.imshow(sub_reg_dist)
    shw.axes.set_title(model)
    bar = plt.colorbar(shw)
    plt.show()

In [ ]:
client.close()